# 10 — Phase 2: Residual Spatial Hybrid Architecture (AML + GNN Fusion)

## Motivation & Engineering Hypothesis
In our benchmark evaluation across the four initial paradigms, we identified a fundamental trade-off:
1. **AML Tree Ensembles (Random Forest / XGBoost)** excel at fine-grained temporal pattern recognition by exploiting localized speed lags with sharp piecewise partitions.
2. **Spatial Graph Neural Networks (GAT)** capture network topology and corridor shockwave propagation, but risk over-smoothing local extreme transitions when trained standalone.

### The Residual Hybrid Hypothesis
Rather than choosing between AML and DL, we deploy a **two-stage hybrid architecture**:
* **Stage 1 (Base Predictor):** A localized AML model (Segment-Wise Random Forest) predicts the base speed $\hat{y}_{s,t}^{\text{base}}$.
* **Residual Error Extraction:** We extract spatial error residuals: $e_{s,t} = y_{s,t} - \hat{y}_{s,t}^{\text{base}}$.
* **Stage 2 (Spatial Graph Compensator):** A PyTorch Graph Attention Network trains on the error residuals to learn propagating network shockwaves.
* **Final Composite Prediction:** $\hat{y}_{s,t}^{\text{final}} = \hat{y}_{s,t}^{\text{base}} + \hat{e}_{s,t}^{\text{GAT}}$.


In [1]:
from pathlib import Path
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.ensemble import RandomForestRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
GRAPH = DATA / "sensor_graph"
RESULTS = ROOT / "results"
METRICS = RESULTS / "metrics"
FIGURES = RESULTS / "figures"
PREDICTIONS = RESULTS / "predictions"

from traffic_prediction.data import load_speed_csv, observed_speeds, prepare_speed_inputs, HORIZON_STEPS
from traffic_prediction.features import time_features
from traffic_prediction.graph import build_distance_adjacency, compute_dual_diffusion_matrices
from traffic_prediction.hybrid import ResidualSpatialGAT, ResidualSpatialHybrid
from traffic_prediction.metrics import regression_metrics

print(f"PyTorch version: {torch.__version__}")
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Compute device: {device}")


PyTorch version: 2.14.1
Compute device: mps


## 1. Load Road Adjacency & Compute Directed Dual Diffusion Operators
We load the METR-LA distance table and compute the thresholded Gaussian distance adjacency ($N=207$ nodes) along with forward ($P_f$) and backward ($P_b$) transition matrices.


In [2]:
raw_df = load_speed_csv(DATA / "METR-LA.csv")
sensor_ids = [str(c) for c in raw_df.columns]
n_sensors = len(sensor_ids)

dist_df = pd.read_csv(GRAPH / "distances_la_2012.csv")
adj = build_distance_adjacency(dist_df, sensor_ids, threshold=0.1)
P_f, P_b = compute_dual_diffusion_matrices(adj)

print(f"Loaded {n_sensors} sensors. Adjacency shape: {adj.shape}, non-zero edges: {(adj > 0).sum()}")
print(f"Forward diffusion P_f shape: {P_f.shape}, Backward diffusion P_b shape: {P_b.shape}")


Loaded 207 sensors. Adjacency shape: (207, 207), non-zero edges: 2833
Forward diffusion P_f shape: (207, 207), Backward diffusion P_b shape: (207, 207)


## 2. Telemetry Ingestion & Chronological Split Protocol
Strict 70% / 10% / 20% chronological split is enforced with causal forward-fill and training-only standardization.


In [3]:
observed = observed_speeds(raw_df)
n_timestamps = len(observed)

train_end = int(n_timestamps * 0.70)
val_end = int(n_timestamps * 0.80)
print(f"Timestamps: {n_timestamps} | Train: 0..{train_end} | Val: {train_end}..{val_end} | Test: {val_end}..{n_timestamps}")

speed_inputs = prepare_speed_inputs(observed, train_end=train_end)
train_mean = speed_inputs[:train_end].mean(axis=0)
train_std = speed_inputs[:train_end].std(axis=0)
train_std[train_std < 1e-4] = 1.0
speed_scaled = (speed_inputs - train_mean) / train_std

clock = time_features(raw_df.index)

horizons = {5: 1, 15: 3, 30: 6, 60: 12}
steps = np.array(list(horizons.values()), dtype=np.int32)
max_step = int(steps.max())
origins = np.arange(5, n_timestamps - max_step, dtype=np.int32)

train_rows = np.flatnonzero(origins + max_step < train_end)
val_rows = np.flatnonzero((origins >= train_end) & (origins + max_step < val_end))
test_rows = np.flatnonzero(origins >= val_end)

print(f"Origins: total={len(origins)}, train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")


Timestamps: 34272 | Train: 0..23990 | Val: 23990..27417 | Test: 27417..34272
Origins: total=34255, train=23973, val=3415, test=6843


## 3. Stage 1: Fit Segment-Wise Random Forest Regressors
We fit 207 segment-wise multi-output Random Forests (one per loop detector) predicting 4 horizons simultaneously.


In [4]:
t0 = time.time()
rf_predictions_all = np.zeros((len(origins), n_sensors, len(steps)), dtype=np.float32)

for s_idx in range(n_sensors):
    X_s = np.column_stack([
        speed_inputs[origins, s_idx],
        speed_inputs[origins - 1, s_idx],
        speed_inputs[origins - 2, s_idx],
        speed_inputs[origins - 5, s_idx],
        clock[origins]
    ]).astype(np.float32)

    target_idx = origins[:, None] + steps[None, :]
    targets_s = observed[target_idx, s_idx]

    train_mask = (target_idx[:, -1] < train_end) & np.isfinite(targets_s).all(axis=1)
    train_sub = np.flatnonzero(train_mask)
    if len(train_sub) > 2000:
        rng = np.random.default_rng(42 + s_idx)
        train_sub = np.sort(rng.choice(train_sub, 2000, replace=False))

    rf = RandomForestRegressor(
        n_estimators=10, max_depth=10, min_samples_leaf=8,
        max_features=1.0, n_jobs=-1, random_state=42
    )
    rf.fit(X_s[train_sub], targets_s[train_sub])
    rf_predictions_all[:, s_idx, :] = rf.predict(X_s).astype(np.float32)

print(f"Trained {n_sensors} segment models in {time.time() - t0:.2f}s.")


Trained 207 segment models in 6.33s.


## 4. Extract Spatial Error Residuals
We compute the difference between actual observed speeds and Random Forest base predictions. Residuals are standardized by sensor standard deviation for neural network training.


In [5]:
target_indices = origins[:, None] + steps[None, :]
actual_targets = observed[target_indices].transpose(0, 2, 1)  # (N_origins, n_sensors, 4)
target_mask = np.isfinite(actual_targets)

# Standardized residuals: R_scaled = (Y - Y_hat_base) / sigma
residuals_scaled = np.where(target_mask, (actual_targets - rf_predictions_all) / train_std[None, :, None], 0.0).astype(np.float32)

rf_test_errors = actual_targets[test_rows] - rf_predictions_all[test_rows]
print("Baseline Random Forest Test MAE across horizons:")
for h_idx, mins in enumerate([5, 15, 30, 60]):
    v = target_mask[test_rows, :, h_idx]
    mae = np.mean(np.abs(rf_test_errors[:, :, h_idx][v]))
    print(f"  RF {mins:02d}m Test MAE: {mae:.3f} mph")


Baseline Random Forest Test MAE across horizons:
  RF 05m Test MAE: 2.519 mph
  RF 15m Test MAE: 3.119 mph
  RF 30m Test MAE: 3.635 mph
  RF 60m Test MAE: 4.192 mph


## 5. Stage 2: Train Residual Spatial Graph Compensator
A Graph Attention Network is trained on the standardized residuals to capture spatial corridor dependencies and upstream/downstream shockwave propagation.


In [6]:
adj_tensor = torch.from_numpy(adj.astype(np.float32))
gat_compensator = ResidualSpatialGAT(
    adjacency=adj_tensor,
    input_size=10,
    hidden_size=32,
    output_size=4,
    dropout=0.1
).to(device)

def make_batch(row_indices):
    t = origins[row_indices]
    history = np.stack([speed_scaled[t - lag] for lag in range(5, -1, -1)], axis=-1)
    clock_b = np.broadcast_to(clock[t, None, :], (len(t), n_sensors, 4))
    x = np.concatenate([history, clock_b], axis=-1)
    return (
        torch.tensor(x, dtype=torch.float32, device=device),
        torch.tensor(residuals_scaled[row_indices], dtype=torch.float32, device=device),
        torch.tensor(target_mask[row_indices], dtype=torch.bool, device=device),
    )

optimizer = torch.optim.Adam(gat_compensator.parameters(), lr=0.003, weight_decay=1e-5)
batch_size = 32
epochs = 10
patience = 3
best_val_loss = float("inf")
best_state = None
stale = 0
training_history = []

rng = np.random.default_rng(42)
for epoch in range(epochs):
    gat_compensator.train()
    shuffled = rng.permutation(train_rows)
    train_loss = train_cnt = 0.0

    for start in range(0, len(shuffled), batch_size):
        b_rows = shuffled[start:start + batch_size]
        bx, by, bmask = make_batch(b_rows)

        optimizer.zero_grad(set_to_none=True)
        pred = gat_compensator(bx)
        diff_sq = (pred - by) ** 2
        loss = (diff_sq * bmask).sum() / bmask.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gat_compensator.parameters(), 2.0)
        optimizer.step()

        train_loss += loss.item() * bmask.sum().item()
        train_cnt += bmask.sum().item()

    gat_compensator.eval()
    val_loss = val_cnt = 0.0
    with torch.no_grad():
        for start in range(0, len(val_rows), batch_size):
            b_rows = val_rows[start:start + batch_size]
            bx, by, bmask = make_batch(b_rows)
            pred = gat_compensator(bx)
            diff_sq = (pred - by) ** 2
            val_loss += (diff_sq * bmask).sum().item()
            val_cnt += bmask.sum().item()

    train_mse = train_loss / train_cnt
    val_mse = val_loss / val_cnt
    training_history.append({"epoch": epoch + 1, "train_mse": train_mse, "val_mse": val_mse})
    print(f"Epoch {epoch+1:02d}: train MSE={train_mse:.4f}, val MSE={val_mse:.4f}")

    if val_mse < best_val_loss:
        best_val_loss = val_mse
        best_state = copy.deepcopy(gat_compensator.state_dict())
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print("Early stopping triggered.")
            break

if best_state is not None:
    gat_compensator.load_state_dict(best_state)


Epoch 01: train MSE=0.3586, val MSE=0.3922
Epoch 02: train MSE=0.3562, val MSE=0.3895
Epoch 03: train MSE=0.3554, val MSE=0.3868
Epoch 04: train MSE=0.3547, val MSE=0.3858
Epoch 05: train MSE=0.3540, val MSE=0.3856
Epoch 06: train MSE=0.3536, val MSE=0.3863
Epoch 07: train MSE=0.3533, val MSE=0.3858
Epoch 08: train MSE=0.3530, val MSE=0.3859
Early stopping triggered.


## 6. Test Evaluation & Composite Forecast Benchmarking
We generate test predictions and combine the base AML speed with the GAT residual adjustment:
$$\hat{y}_{s,t}^{\text{final}} = \hat{y}_{s,t}^{\text{base}} + \hat{e}_{s,t}^{\text{GAT}}$$


In [7]:
gat_compensator.eval()
pred_res_batches = []
with torch.no_grad():
    for start in range(0, len(test_rows), batch_size):
        b_rows = test_rows[start:start + batch_size]
        bx, _, _ = make_batch(b_rows)
        pred_res_batches.append(gat_compensator(bx).cpu().numpy())

pred_residuals_scaled = np.concatenate(pred_res_batches, axis=0)
pred_residuals = pred_residuals_scaled * train_std[None, :, None]

base_test_pred = rf_predictions_all[test_rows]
hybrid_test_pred = base_test_pred + pred_residuals

actual_test = actual_targets[test_rows]
test_target_times = target_indices[test_rows]
target_hours = raw_df.index.hour.to_numpy()[test_target_times]
target_weekdays = raw_df.index.day_of_week.to_numpy()[test_target_times]
peak_mask = (target_weekdays < 5) & (((target_hours >= 7) & (target_hours < 10)) | ((target_hours >= 16) & (target_hours < 19)))

period_masks = {
    "All": np.ones_like(peak_mask, dtype=bool),
    "Peak": peak_mask,
    "Off-peak": ~peak_mask,
}

hybrid_scores = []
for h_idx, minutes in enumerate([5, 15, 30, 60]):
    for period_name, p_mask in period_masks.items():
        valid = np.isfinite(actual_test[:, :, h_idx]) & p_mask[:, h_idx, None]
        err = actual_test[:, :, h_idx][valid] - hybrid_test_pred[:, :, h_idx][valid]
        act = actual_test[:, :, h_idx][valid]
        mae = float(np.mean(np.abs(err)))
        mape = float(np.mean(np.abs(err) / act) * 100)
        rmse = float(np.sqrt(np.mean(err ** 2)))
        cnt = int(valid.sum())
        hybrid_scores.append({
            "dataset": "METR-LA",
            "model": "Residual Hybrid",
            "period": period_name,
            "horizon_minutes": minutes,
            "MAE": mae,
            "MAPE_percent": mape,
            "RMSE": rmse,
            "valid_test_values": cnt,
        })

hybrid_df = pd.DataFrame(hybrid_scores)
hybrid_csv = METRICS / "metr_la_hybrid_metrics.csv"
hybrid_df.to_csv(hybrid_csv, index=False)
print(f"Saved hybrid metrics to: {hybrid_csv}")

display_df = hybrid_df[hybrid_df["period"] == "All"][["model", "horizon_minutes", "MAE", "RMSE", "MAPE_percent"]].round(3)
print(display_df.to_string(index=False))


Saved hybrid metrics to: /Users/gaurav_meena/Desktop/AI/intelligent-traffic-forecasting/results/metrics/metr_la_hybrid_metrics.csv
          model  horizon_minutes   MAE  RMSE  MAPE_percent
Residual Hybrid                5 2.478 4.239         6.140
Residual Hybrid               15 3.079 5.654         8.202
Residual Hybrid               30 3.592 6.634        10.112
Residual Hybrid               60 4.132 7.547        12.279


## 7. Save Sample Trajectory & Comparative Visualizations
We export the 15-minute sample forecast for sensor 773869 and generate comparative horizon benchmark plots.


In [8]:
sample_sensor_idx = len(sensor_ids) // 2
sample_df = pd.DataFrame({
    "timestamp": raw_df.index[target_indices[test_rows, 1]], # 15m
    "actual_speed": actual_test[:, sample_sensor_idx, 1],
    "base_rf_speed_15min": base_test_pred[:, sample_sensor_idx, 1],
    "gat_residual_15min": pred_residuals[:, sample_sensor_idx, 1],
    "predicted_speed_15min": hybrid_test_pred[:, sample_sensor_idx, 1],
})
sample_csv = PREDICTIONS / "metr_la_hybrid_sample_sensor.csv"
sample_df.to_csv(sample_csv, index=False)
print(f"Saved sample trajectory to {sample_csv}")

comp_df = pd.read_csv(METRICS / "metr_la_model_comparison.csv")
comp_filtered = comp_df[comp_df["model"] != "Residual Hybrid"]
updated_comp = pd.concat([comp_filtered, hybrid_df], ignore_index=True)
updated_comp.to_csv(METRICS / "metr_la_model_comparison.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {
    "Persistence": "#94a3b8",
    "Random Forest": "#0f766e",
    "XGBoost": "#d97706",
    "GAT": "#2563eb",
    "Residual Hybrid": "#7c3aed"
}

for idx, period in enumerate(["All", "Peak", "Off-peak"]):
    ax = axes[idx]
    view = updated_comp[updated_comp["period"] == period]
    for model_name, grp in view.groupby("model"):
        grp_sorted = grp.sort_values("horizon_minutes")
        ax.plot(
            grp_sorted["horizon_minutes"],
            grp_sorted["MAE"],
            marker="o",
            linewidth=2.2,
            label=model_name,
            color=palette.get(model_name, "#64748b")
        )
    ax.set_title(f"Test MAE by Horizon ({period})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Forecast Horizon (minutes)", fontsize=10)
    ax.set_ylabel("Masked MAE (mph)", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_xticks([5, 15, 30, 60])
    if idx == 0:
        ax.legend(frameon=True)

plt.tight_layout()
fig_path = FIGURES / "30_residual_hybrid_model_comparison.png"
plt.savefig(fig_path, dpi=200)
plt.close()
print(f"Saved comparative figure to: {fig_path}")


Saved sample trajectory to /Users/gaurav_meena/Desktop/AI/intelligent-traffic-forecasting/results/predictions/metr_la_hybrid_sample_sensor.csv
Saved comparative figure to: /Users/gaurav_meena/Desktop/AI/intelligent-traffic-forecasting/results/figures/30_residual_hybrid_model_comparison.png
